> **Version corrigée** — ce notebook contient le code complet de tous les exercices, exécuté de bout en bout, ainsi qu'un élément de réponse pour chaque question d'observation. La version étudiant (à compléter soi-même) est téléchargeable depuis la page du cours.

# k plus proches voisins et pertes de substitution

**Notebook 4/9 — Introduction to Supervised Machine Learning**
*Guillaume Metzler — Université Lyon 2 (L3 → Master)*

Ce notebook regroupe deux notions qui n'ont l'air de rien en commun au
premier abord, mais qui se recoupent en fin de parcours :

- les **pertes de substitution** (*surrogate losses*), qui remplacent la
  perte 0-1 par une fonction convexe pour rendre l'apprentissage
  optimisable (hinge, logistique, exponentielle) ;
- les **k plus proches voisins** (k-NN), une méthode de classification
  non paramétrique fondée uniquement sur une notion de distance, sans
  aucune perte à minimiser.

Von Luxburg et Bousquet (2004) ont montré un lien étroit entre le 1-NN et
le SVM étudié dans le prochain notebook — un bon aperçu de ce qui nous
attend.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs, make_circles, make_classification
from sklearn.datasets import load_wine, load_digits, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score

np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 5)


## 1. Pertes de substitution : remplacer la perte 0-1

En classification binaire, avec $y \in \{-1, +1\}$ et un score
$h(x) \in \mathbb{R}$ (par exemple $h(x) = \langle w, x \rangle$ pour un
modèle linéaire), l'erreur naturelle est la **perte 0-1** :
$$ \ell_{0-1}(h(x), y) = \mathbb{1}\{y\,h(x) \leq 0\}, $$
exprimée en fonction de la **marge** $u = y\,h(x)$ : la prédiction est
correcte si et seulement si $u > 0$.

Vue comme fonction de $u$, cette perte est **non convexe** et **non
différentiable** en $u=0$ : minimiser le risque empirique associé est un
problème combinatoire, hors de portée d'une descente de gradient. On la
remplace donc par une **perte de substitution** convexe qui la majore (à
peu près) et que l'on sait optimiser :

- **hinge loss** (SVM) : $\ell_{\text{hinge}}(u) = \max(0,\ 1-u)$, nulle
  dès que $u \geq 1$ ;
- **perte exponentielle** (boosting) : $\ell_{\exp}(u) = \exp(-u)$, ne
  s'annule jamais ;
- **perte logistique** (régression logistique) : $\ell_{\text{log}}(u) =
  \frac{1}{\ln 2}\ln(1+\exp(-u))$, normalisée pour valoir $1$ en $u=0$.

In [ ]:
def perte_01(u):
    '''Perte 0-1 en fonction de la marge u = y * h(x).'''
    return (u <= 0).astype(float)

def perte_hinge(u):
    '''Hinge loss (SVM).'''
    return np.maximum(0.0, 1.0 - u)

def perte_exponentielle(u):
    '''Perte exponentielle (boosting).'''
    return np.exp(-u)

def perte_logistique(u):
    '''Perte logistique (regression logistique), normalisee par 1/ln(2).'''
    return np.log1p(np.exp(-u)) / np.log(2)

u = np.linspace(-3, 3, 400)

fig, ax = plt.subplots()
ax.plot(u, perte_01(u), label="Perte 0-1", color="black", linewidth=2)
ax.plot(u, perte_hinge(u), label="Hinge loss (SVM)", color="tab:blue")
ax.plot(u, perte_logistique(u), label="Perte logistique", color="tab:orange")
ax.plot(u, perte_exponentielle(u), label="Perte exponentielle (boosting)", color="tab:green")
ax.axvline(0, color="gray", linestyle=":", linewidth=1)
ax.set_xlabel("Marge $u = y \\cdot h(x)$")
ax.set_ylabel("Valeur de la perte")
ax.set_title("Perte 0-1 et pertes de substitution en fonction de la marge")
ax.set_ylim(0, 3)
ax.legend()
plt.show()


$$ $$

**Question 1 :** Parmi les quatre courbes tracées, lesquelles sont convexes ? Laquelle n'est différentiable partout, y compris en $u=0$ ? Et laquelle des trois pertes de substitution pénalise le plus fortement un exemple très mal classé (marge $u$ très négative, à gauche du graphique) ?

$$ $$

*Éléments de réponse.* Les trois pertes de substitution (hinge, logistique, exponentielle) sont convexes en $u$ ; la perte 0-1 ne l'est pas. La perte logistique est la seule différentiable partout (la hinge a un point anguleux en $u=1$). Pour $u$ très négatif, la perte exponentielle croît beaucoup plus vite que les deux autres (croissance exponentielle contre croissance linéaire ou logarithmique) : elle est donc la plus sensible aux exemples très mal classés, ce qui la rend aussi la plus vulnérable aux points aberrants (un défaut connu du boosting basé sur AdaBoost).

In [ ]:
# Un exemple d'utilisation : evaluer perte_01 et perte_logistique
# sur quelques couples (y, h(x)).
y_demo = np.array([1, 1, -1])
h_demo = np.array([2.0, 0.0, 0.5])
u_demo = y_demo * h_demo

for y_i, h_i, u_i in zip(y_demo, h_demo, u_demo):
    print(f"y={y_i:+d}, h(x)={h_i:+.2f}, marge u={u_i:+.2f} -> "
          f"perte_01={perte_01(np.array([u_i]))[0]:.4f}, "
          f"perte_logistique={perte_logistique(np.array([u_i]))[0]:.4f}")


### Exercice 1

En reprenant la même mécanique, écrivez deux fonctions `hinge(y, h)` et
`exponentielle(y, h)` qui calculent respectivement la hinge loss et la
perte exponentielle (`y` et `h` sont des tableaux numpy de même taille).
Évaluez-les sur les couples $(y, h(x))$ ci-dessous et vérifiez vos
résultats par un `assert` :

| $y$ | $h(x)$ |
|---|---|
| $1$ | $0.5$ |
| $1$ | $1.0$ |
| $-1$ | $1.0$ |
| $-1$ | $-2.0$ |

In [ ]:
y_vals = np.array([1, 1, -1, -1])
h_vals = np.array([0.5, 1.0, 1.0, -2.0])

def hinge(y, h):
    return np.maximum(0.0, 1.0 - y * h)

def exponentielle(y, h):
    return np.exp(-y * h)

pertes_hinge = hinge(y_vals, h_vals)
pertes_exp = exponentielle(y_vals, h_vals)
for y_i, h_i, lh, le in zip(y_vals, h_vals, pertes_hinge, pertes_exp):
    print(f"y={y_i:+d}, h(x)={h_i:+.1f} -> hinge={lh:.4f}, exponentielle={le:.4f}")

# Verification : marges u = y*h correspondantes = 0.5, 1.0, -1.0, 2.0
attendu_hinge = np.array([0.5, 0.0, 2.0, 0.0])
attendu_exp = np.array([0.6065, 0.3679, 2.7183, 0.1353])
assert np.allclose(pertes_hinge, attendu_hinge, atol=1e-3)
assert np.allclose(pertes_exp, attendu_exp, atol=1e-3)
print("\nOK : les valeurs calculees correspondent a celles du graphique.")


## 2. k plus proches voisins : principe et choix de k

Le k-NN [Cover et Hart, 1967] est une méthode **non paramétrique** : elle
ne fait aucune hypothèse sur la distribution des données. Pour prédire
l'étiquette d'une nouvelle observation $x'$, on calcule sa distance à
chaque exemple d'apprentissage, on garde les $k$ plus proches, et on
prédit l'étiquette **majoritaire** parmi eux :
$$ \hat{y}(x') = \arg\max_{y \in \mathcal{Y}} k_y, $$
où $k_y$ est le nombre de voisins de classe $y$ parmi les $k$ retenus.

Cover et Hart (1967) ont montré que pour $k=1$, lorsque $m$ est
suffisamment grand, l'erreur du 1-NN est au plus le double de l'erreur
de Bayes (l'erreur minimale atteignable compte tenu de la distribution
des données). La distance la plus courante est la distance euclidienne
($L_2$) : $d_2(x,x') = \sqrt{\sum_{j=1}^d (x_j-x'_j)^2}$.

In [ ]:
# Jeu d'apprentissage 1D et une requete x' = 4.
X_train_1d = np.array([1, 2, 3, 5, 8]).reshape(-1, 1)
y_train_1d = np.array([1, 1, 0, 0, 1])
x_query = 4

distances = np.abs(X_train_1d.ravel() - x_query)
ordre = np.argsort(distances)
print("Points tries par distance croissante a x' = 4 :")
for idx in ordre:
    print(f"  x={X_train_1d[idx, 0]}, y={y_train_1d[idx]}, distance={distances[idx]}")

for k in [1, 3, 5]:
    voisins = ordre[:k]
    votes = y_train_1d[voisins]
    classes, effectifs = np.unique(votes, return_counts=True)
    prediction = classes[np.argmax(effectifs)]
    print(f"k={k} -> voisins retenus: {sorted(X_train_1d[voisins].ravel())}, "
          f"vote majoritaire = {prediction}")


### Exercice 2

On considère un autre jeu 1D : `X_train_1d2 = [0, 2, 3, 7, 9]`,
`y_train_1d2 = [0, 0, 1, 1, 0]`, et trois requêtes
`X_query2 = [[1], [5], [8.5]]`. À l'aide de
`sklearn.neighbors.KNeighborsClassifier`, entraînez un classifieur avec
`n_neighbors=1` puis un second avec `n_neighbors=3`, prédisez les
étiquettes des trois requêtes pour chaque k, et comparez au calcul « à la
main » (par distance croissante, comme dans la démonstration).

In [ ]:
X_train_1d2 = np.array([0, 2, 3, 7, 9]).reshape(-1, 1)
y_train_1d2 = np.array([0, 0, 1, 1, 0])
X_query2 = np.array([[1], [5], [8.5]])

clf_k1 = KNeighborsClassifier(n_neighbors=1).fit(X_train_1d2, y_train_1d2)
clf_k3 = KNeighborsClassifier(n_neighbors=3).fit(X_train_1d2, y_train_1d2)

pred_k1 = clf_k1.predict(X_query2)
pred_k3 = clf_k3.predict(X_query2)

print("Predictions k=1 :", pred_k1)
print("Predictions k=3 :", pred_k3)

# Verification a la main :
# x'=1   -> voisins tries: 0(d1),2(d1),3(d2),7(d6),9(d8)
#           k=1: 0 (tie de distance mais meme label 0) ; k=3: [0,0,1] -> 0
# x'=5   -> voisins tries: 3(d2),7(d2),9(d4),2(d3),0(d5)
#           k=1: 1 (tie de distance mais meme label 1) ; k=3: [1,1,0] -> 1
# x'=8.5 -> voisins tries: 9(d0.5),7(d1.5),3(d5.5),2(d6.5),0(d8.5)
#           k=1: 0 ; k=3: [0,1,1] -> 1
attendu_k1 = np.array([0, 1, 0])
attendu_k3 = np.array([0, 1, 1])
assert np.array_equal(pred_k1, attendu_k1)
assert np.array_equal(pred_k3, attendu_k3)
print("\nOK : les predictions de KNeighborsClassifier correspondent au calcul manuel.")


### Effet de k sur la frontière de décision

Regardons ce qui se passe sur un jeu non linéairement séparable
(`make_moons`, avec du bruit) lorsqu'on fait varier $k$.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=0)

xx, yy = np.meshgrid(
    np.linspace(X_moons[:, 0].min() - 0.5, X_moons[:, 0].max() + 0.5, 300),
    np.linspace(X_moons[:, 1].min() - 0.5, X_moons[:, 1].max() + 0.5, 300),
)

valeurs_k = [1, 5, 15, 50]
fig, axes = plt.subplots(1, len(valeurs_k), figsize=(18, 4.5), sharex=True, sharey=True)

for ax, k in zip(axes, valeurs_k):
    clf = KNeighborsClassifier(n_neighbors=k).fit(X_moons, y_moons)
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="coolwarm",
               edgecolor="k", s=15)
    ax.set_title(f"k = {k}")
    ax.set_xlabel("$x_1$")
axes[0].set_ylabel("$x_2$")
fig.suptitle("Frontieres de decision du k-NN selon k (make_moons)")
plt.tight_layout()
plt.show()


$$ $$

**Question 2 :** Décrivez la frontière obtenue avec `k=1` par rapport à celle obtenue avec `k=50`. Quel phénomène (sous- ou sur-apprentissage) illustre chaque extrême ?

$$ $$

$$ $$

**Question 3 :** Quelle valeur de k, parmi les quatre testées, offre selon vous le meilleur compromis sur cet exemple ? Comment reliez-vous, intuitivement, la valeur de k à la variance du modèle ?

$$ $$

*Éléments de réponse.* Avec `k=1`, la frontière colle à chaque point, y compris au bruit : c'est du sur-apprentissage, la variance est maximale. Avec `k=50`, la frontière est presque lisse et ne capture plus la forme en croissants des données : c'est du sous-apprentissage, le biais est élevé.

*Éléments de réponse.* Ici `k=5` ou `k=15` donnent un compromis raisonnable : la frontière suit la structure des deux croissants sans se laisser dévier par chaque point isolé. Plus k est petit, plus chaque prédiction dépend d'un tout petit nombre de points (grande variance d'un jeu d'apprentissage à l'autre) ; plus k est grand, plus la prédiction moyenne un grand nombre de voisins et se stabilise (variance plus faible, mais biais plus fort).

In [ ]:
# Choix de k par validation croisee (5 plis), sur les donnees make_moons.
valeurs_k_cv = list(range(1, 41))
scores_moons = []

cv = KFold(n_splits=5, shuffle=True, random_state=0)
for k in valeurs_k_cv:
    clf = KNeighborsClassifier(n_neighbors=k)
    scores_moons.append(cross_val_score(clf, X_moons, y_moons, cv=cv).mean())

plt.figure()
plt.plot(valeurs_k_cv, scores_moons, marker="o", markersize=3)
plt.xlabel("k (nombre de voisins)")
plt.ylabel("Exactitude moyenne (validation croisee 5 plis)")
plt.title("Choix de k par validation croisee (make_moons)")
plt.show()

meilleur_k_moons = valeurs_k_cv[int(np.argmax(scores_moons))]
print("Meilleur k :", meilleur_k_moons, "-- exactitude :", round(max(scores_moons), 4))


### Exercice 3

Sur le jeu `load_wine` (standardisé avec `StandardScaler`, séparé en
train/test), trouvez la valeur de k optimale par validation croisée
(5 plis) parmi `range(1, 31)`. Tracez la courbe de l'exactitude moyenne
en fonction de k, indiquez le meilleur k, puis entraînez un
`KNeighborsClassifier` avec ce k sur le train et évaluez-le sur le test.
Commentez brièvement (dans un `print`) le compromis biais-variance
illustré par la courbe.

In [ ]:
X_wine, y_wine = load_wine(return_X_y=True)
X_train_wn, X_test_wn, y_train_wn, y_test_wn = train_test_split(
    X_wine, y_wine, test_size=0.3, random_state=0, stratify=y_wine)

scaler_wn = StandardScaler().fit(X_train_wn)
X_train_wn_sc = scaler_wn.transform(X_train_wn)
X_test_wn_sc = scaler_wn.transform(X_test_wn)

valeurs_k_wn = list(range(1, 31))
scores_wn = []
cv_wn = KFold(n_splits=5, shuffle=True, random_state=0)
for k in valeurs_k_wn:
    clf = KNeighborsClassifier(n_neighbors=k)
    scores_wn.append(cross_val_score(clf, X_train_wn_sc, y_train_wn, cv=cv_wn).mean())

plt.figure()
plt.plot(valeurs_k_wn, scores_wn, marker="o", markersize=3)
plt.xlabel("k (nombre de voisins)")
plt.ylabel("Exactitude moyenne (validation croisee 5 plis)")
plt.title("Choix de k par validation croisee (load_wine, standardise)")
plt.show()

meilleur_k_wn = valeurs_k_wn[int(np.argmax(scores_wn))]
clf_final_wn = KNeighborsClassifier(n_neighbors=meilleur_k_wn)
clf_final_wn.fit(X_train_wn_sc, y_train_wn)
acc_test_wn = accuracy_score(y_test_wn, clf_final_wn.predict(X_test_wn_sc))

print("Meilleur k :", meilleur_k_wn, "-- exactitude test :", round(acc_test_wn, 4))
# Comme sur make_moons : un k trop petit sur-apprend (variance elevee), un k
# trop grand sous-apprend (biais eleve) ; la validation croisee permet de
# choisir un compromis raisonnable sans toucher au jeu de test.


## 3. Distance, mise à l'échelle et fléau de la dimension

Le k-NN repose entièrement sur la distance choisie. Un premier problème :
si les descripteurs ont des échelles très différentes, la distance
euclidienne est **dominée** par celui de plus grande amplitude, même s'il
n'est pas le plus informatif. Exemple, avec
$x^{(1)}=(0.3,\ 0.89,\ 232)$ et $x^{(2)}=(-0.1,\ 0.23,\ 652)$ :
$$ d_2^2(x^{(1)}, x^{(2)}) = (0.3+0.1)^2 + (0.89-0.23)^2 + (232-652)^2
\approx (232-652)^2, $$
les deux premiers descripteurs n'ont quasiment aucun poids. On corrige
cela en mettant les variables à l'échelle, par exemple par
**normalisation** (centrage-réduction) $v_j \leftarrow (v_j-\mu_j)/\sigma_j$
— c'est ce que fait `StandardScaler`.

In [ ]:
wine = load_wine()
X_wine_raw, y_wine_raw = wine.data, wine.target

print("Min/Max de quelques variables de load_wine :")
for name, col in list(zip(wine.feature_names, X_wine_raw.T))[:2] + [
        (wine.feature_names[-1], X_wine_raw[:, -1])]:
    print(f"  {name:25s}: min={col.min():8.2f}, max={col.max():8.2f}")

X_train_wr, X_test_wr, y_train_wr, y_test_wr = train_test_split(
    X_wine_raw, y_wine_raw, test_size=0.3, random_state=0, stratify=y_wine_raw)

clf_brut = KNeighborsClassifier(n_neighbors=5).fit(X_train_wr, y_train_wr)
acc_brut = accuracy_score(y_test_wr, clf_brut.predict(X_test_wr))

scaler_wr = StandardScaler().fit(X_train_wr)
clf_scale = KNeighborsClassifier(n_neighbors=5).fit(
    scaler_wr.transform(X_train_wr), y_train_wr)
acc_scale = accuracy_score(y_test_wr, clf_scale.predict(scaler_wr.transform(X_test_wr)))

print(f"\nExactitude k-NN (k=5) SANS mise a l'echelle : {acc_brut:.3f}")
print(f"Exactitude k-NN (k=5) AVEC StandardScaler    : {acc_scale:.3f}")

plt.figure()
plt.bar(["Sans mise a l'echelle", "Avec StandardScaler"], [acc_brut, acc_scale],
        color=["tab:red", "tab:blue"])
plt.ylabel("Exactitude sur le jeu de test")
plt.ylim(0, 1)
plt.title("Effet de la mise a l'echelle sur le k-NN (load_wine)")
plt.show()


$$ $$

**Question 4 :** Pourquoi la mise à l'échelle améliore-t-elle nettement les performances ici ? Quel type de descripteur (regardez les min/max affichés) domine le calcul de la distance sans elle ?

$$ $$

*Éléments de réponse.* Sans mise à l'échelle, la distance euclidienne est dominée par les descripteurs de grande amplitude, ici typiquement `proline` (valeurs de l'ordre de plusieurs centaines) alors que d'autres descripteurs comme `hue` varient entre 0 et 2 environ. Ces derniers pèsent alors quasiment pour rien dans le calcul, même s'ils sont informatifs. Après `StandardScaler`, chaque descripteur a la même variance, donc un poids comparable dans la distance.

On peut aussi changer de distance : la distance de **Manhattan** ($L_1$)
$d_1(x,x') = \sum_j |x_j-x'_j|$ est moins sensible aux grands écarts sur
une seule variable que la distance euclidienne ($L_2$), qui les élève au
carré. `KNeighborsClassifier` accepte l'argument `metric`.

In [ ]:
X_circ, y_circ = make_circles(n_samples=200, noise=0.15, factor=0.4, random_state=2)

xx_c, yy_c = np.meshgrid(
    np.linspace(X_circ[:, 0].min() - 0.3, X_circ[:, 0].max() + 0.3, 250),
    np.linspace(X_circ[:, 1].min() - 0.3, X_circ[:, 1].max() + 0.3, 250),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, metric in zip(axes, ["euclidean", "manhattan"]):
    clf = KNeighborsClassifier(n_neighbors=15, metric=metric).fit(X_circ, y_circ)
    Z = clf.predict(np.c_[xx_c.ravel(), yy_c.ravel()]).reshape(xx_c.shape)
    ax.contourf(xx_c, yy_c, Z, alpha=0.3, cmap="coolwarm")
    ax.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ, cmap="coolwarm", edgecolor="k", s=15)
    ax.set_title(f"metric = '{metric}'")
plt.tight_layout()
plt.show()


### Exercice 4

Sur `load_digits` (les descripteurs sont déjà sur une échelle homogène,
des niveaux de gris entre 0 et 16), comparez l'exactitude sur un jeu de
test d'un `KNeighborsClassifier(n_neighbors=5)` avec `metric='euclidean'`
puis avec `metric='manhattan'`.

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
X_train_dig, X_test_dig, y_train_dig, y_test_dig = train_test_split(
    X_dig, y_dig, test_size=0.3, random_state=0, stratify=y_dig)

clf_eucl = KNeighborsClassifier(n_neighbors=5, metric="euclidean")
clf_eucl.fit(X_train_dig, y_train_dig)
acc_eucl = accuracy_score(y_test_dig, clf_eucl.predict(X_test_dig))

clf_manh = KNeighborsClassifier(n_neighbors=5, metric="manhattan")
clf_manh.fit(X_train_dig, y_train_dig)
acc_manh = accuracy_score(y_test_dig, clf_manh.predict(X_test_dig))

print(f"Exactitude (metric='euclidean') : {acc_eucl:.4f}")
print(f"Exactitude (metric='manhattan') : {acc_manh:.4f}")


### Le fléau de la dimension

Le k-NN suppose que la notion de « proximité » reste informative. Or, en
grande dimension, les distances entre points tirés aléatoirement ont
tendance à se **concentrer** : la différence entre le point le plus
proche et le plus lointain devient négligeable devant la distance
elle-même. On tire $n$ points uniformément dans $[0,1]^d$, on mesure la
distance du centre de l'hypercube à chacun, et on regarde le ratio
$(d_{\max}-d_{\min})/d_{\min}$ selon $d$.

In [ ]:
rng = np.random.default_rng(0)
n_points = 1000
dimensions = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
ratios = []

for d in dimensions:
    X_cube = rng.uniform(0, 1, size=(n_points, d))
    requete = np.full(d, 0.5)
    distances_cube = np.linalg.norm(X_cube - requete, axis=1)
    d_min, d_max = distances_cube.min(), distances_cube.max()
    ratios.append((d_max - d_min) / d_min)

plt.figure()
plt.plot(dimensions, ratios, marker="o")
plt.xscale("log")
plt.xlabel("Dimension d (echelle log)")
plt.ylabel("$(d_{max} - d_{min}) / d_{min}$")
plt.title("Fleau de la dimension : effondrement du contraste des distances")
plt.show()


$$ $$

**Question 5 :** Comment interprétez-vous l'effondrement de ce ratio quand $d$ augmente ? Quelle conséquence cela a-t-il, selon vous, pour la notion de « voisin proche » utilisée par le k-NN ?

$$ $$

*Éléments de réponse.* En petite dimension, le point le plus proche et le point le plus lointain sont très différents (ratio élevé) ; en grande dimension, tous les points finissent à peu près à la même distance du point de requête (ratio proche de 0). La notion même de « voisin proche » perd alors son sens : le k-NN (et toute méthode fondée sur les distances) devient peu fiable en très grande dimension, surtout si de nombreux descripteurs ne sont pas informatifs.

### Exercice 5 (Master)

On veut observer l'effet du fléau de la dimension non plus sur un ratio
de distances, mais directement sur la **performance** du k-NN. Générez,
pour chaque valeur de `n_features` dans `[2, 5, 10, 20, 50, 100, 200]`,
un jeu `make_classification(n_samples=500, n_informative=2,
n_redundant=0, n_features=n_features, n_clusters_per_class=1,
random_state=0)` : seules 2 dimensions sont réellement informatives,
toutes les autres sont du bruit. Calculez, pour un
`KNeighborsClassifier(n_neighbors=10)`, l'exactitude moyenne en
validation croisée (5 plis), et tracez-la en fonction de `n_features`
(axe x en échelle log).

In [ ]:
liste_n_features = [2, 5, 10, 20, 50, 100, 200]
scores_dim = []

for n_features in liste_n_features:
    X_dim, y_dim = make_classification(
        n_samples=500, n_features=n_features, n_informative=2, n_redundant=0,
        n_clusters_per_class=1, flip_y=0.01, random_state=0)
    clf = KNeighborsClassifier(n_neighbors=10)
    scores_dim.append(cross_val_score(clf, X_dim, y_dim, cv=5).mean())

plt.figure()
plt.plot(liste_n_features, scores_dim, marker="o")
plt.xscale("log")
plt.xlabel("Nombre total de descripteurs (2 informatifs, le reste du bruit)")
plt.ylabel("Exactitude moyenne (validation croisee 5 plis)")
plt.title("Effet de la dimension sur les performances du k-NN")
plt.show()

print("Exactitude pour chaque nombre de descripteurs :")
for nf, s in zip(liste_n_features, scores_dim):
    print(f"  n_features={nf:>4d} -> exactitude={s:.4f}")
# Avec seulement du bruit ajoute, les performances se degradent nettement
# quand le nombre total de descripteurs augmente : la distance euclidienne
# est de plus en plus dominee par les dimensions non informatives.


## 4. Approfondissement : implémentation et raffinements du k-NN

Pour prédire l'étiquette d'une nouvelle observation, le k-NN doit garder
**tout** le jeu d'apprentissage en mémoire et calculer sa distance à
**chacun** de ses points : pour $n$ exemples de dimension $d$, la
complexité en temps est en $O(nd+nk)$ et la mémoire en $O(n)$. Voyons
d'abord une implémentation « à la main », volontairement naïve.

In [ ]:
def knn_from_scratch(X_train, y_train, X_test, k):
    '''Implementation naive du k-NN (une boucle sur les requetes).'''
    predictions = []
    for x_q in X_test:
        distances = np.linalg.norm(X_train - x_q, axis=1)
        voisins = np.argsort(distances)[:k]
        votes = y_train[voisins]
        classes, effectifs = np.unique(votes, return_counts=True)
        predictions.append(classes[np.argmax(effectifs)])
    return np.array(predictions)

X_toy, y_toy = make_blobs(n_samples=60, centers=2, cluster_std=1.5, random_state=3)
X_toy_train, X_toy_test, y_toy_train, y_toy_test = train_test_split(
    X_toy, y_toy, test_size=0.3, random_state=3)

pred_scratch_toy = knn_from_scratch(X_toy_train, y_toy_train, X_toy_test, k=5)
pred_sklearn_toy = KNeighborsClassifier(n_neighbors=5).fit(
    X_toy_train, y_toy_train).predict(X_toy_test)

print("Predictions identiques a sklearn :", np.array_equal(pred_scratch_toy, pred_sklearn_toy))
assert np.array_equal(pred_scratch_toy, pred_sklearn_toy)


### Exercice 6 (Master)

`knn_from_scratch` ci-dessus contient encore une boucle Python sur les
points de requête. Écrivez une version `knn_vectorise(X_train, y_train,
X_test, k)` dont le **calcul des distances** est entièrement vectorisé
(pas de boucle Python pour cette partie) : construisez directement la
matrice des distances de forme `(n_test, n_train)` par *broadcasting*
(par exemple `X_test[:, None, :] - X_train[None, :, :]`), puis
sélectionnez les k plus proches voisins et le vote majoritaire pour
chaque requête. Vérifiez, sur `load_breast_cancer` standardisé (`k=7`),
que vos prédictions coïncident exactement avec celles de
`knn_from_scratch` et de `KNeighborsClassifier`. Comparez au passage le
temps d'exécution des deux implémentations.

In [ ]:
def knn_vectorise(X_train, y_train, X_test, k):
    # Matrice des distances (n_test, n_train), calculee sans boucle explicite.
    D = np.linalg.norm(X_test[:, None, :] - X_train[None, :, :], axis=2)
    voisins = np.argsort(D, axis=1)[:, :k]
    votes = y_train[voisins]
    return np.array([np.bincount(v).argmax() for v in votes])

cancer = load_breast_cancer()
X_can_train, X_can_test, y_can_train, y_can_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, random_state=0, stratify=cancer.target)
scaler_can = StandardScaler().fit(X_can_train)
X_can_train_sc = scaler_can.transform(X_can_train)
X_can_test_sc = scaler_can.transform(X_can_test)

k = 7
t0 = time.perf_counter()
pred_scratch = knn_from_scratch(X_can_train_sc, y_can_train, X_can_test_sc, k)
t_scratch = time.perf_counter() - t0

t0 = time.perf_counter()
pred_vect = knn_vectorise(X_can_train_sc, y_can_train, X_can_test_sc, k)
t_vect = time.perf_counter() - t0

pred_sklearn = KNeighborsClassifier(n_neighbors=k).fit(
    X_can_train_sc, y_can_train).predict(X_can_test_sc)

assert np.array_equal(pred_scratch, pred_vect)
assert np.array_equal(pred_vect, pred_sklearn)
print("OK : knn_from_scratch, knn_vectorise et KNeighborsClassifier sont d'accord.")
print(f"Temps knn_from_scratch (boucle sur les requetes) : {t_scratch*1000:.2f} ms")
print(f"Temps knn_vectorise (distances vectorisees)      : {t_vect*1000:.2f} ms")


Dans le vote majoritaire « classique », chaque voisin compte pour une
voix, quelle que soit sa distance à la requête. Le **k-NN pondéré**
[Dudani, 1976] pondère chaque voisin par l'inverse de sa distance
($1/d_i$) — c'est l'option `weights='distance'` de `KNeighborsClassifier`
(`'uniform'` par défaut).

**Exemple.** Pour classer $x'$ avec le 3-NN, voisins d'étiquettes
$y_1=1, y_2=1, y_3=-1$ à des distances $d_1=3, d_2=4, d_3=1$ : le vote
non pondéré donne 2 voix à la classe $1$ contre $1$ voix à $-1$, donc
prédit $1$. Pondéré par $1/d_i$, la classe $1$ reçoit
$1/3+1/4=7/12\approx0.58$ et la classe $-1$ reçoit $1/1=1$ : c'est
$-1$ qui l'emporte.

In [ ]:
# Verification numerique de l'exemple ci-dessus.
labels_ex = np.array([1, 1, -1])
distances_ex = np.array([3.0, 4.0, 1.0])

vote_non_pondere = {c: np.sum(labels_ex == c) for c in np.unique(labels_ex)}
poids = 1.0 / distances_ex
vote_pondere = {c: poids[labels_ex == c].sum() for c in np.unique(labels_ex)}

print("Vote non pondere :", vote_non_pondere)
print("Vote pondere (poids 1/distance) :", vote_pondere)
print("Prediction non ponderee :", max(vote_non_pondere, key=vote_non_pondere.get))
print("Prediction ponderee     :", max(vote_pondere, key=vote_pondere.get))


In [ ]:
# Un jeu 2D desequilibre, avec quelques points bruites pres de la frontiere
# entre les deux classes (fourni tel quel).
X_imb, y_imb = make_blobs(
    n_samples=[300, 40], centers=[(0, 0), (2.5, 2.5)],
    cluster_std=[1.3, 1.3], random_state=1)
rng_imb = np.random.default_rng(1)
points_bruit = rng_imb.uniform(0.8, 1.8, size=(15, 2))
labels_bruit = rng_imb.integers(0, 2, size=15)
X_imb = np.vstack([X_imb, points_bruit])
y_imb = np.concatenate([y_imb, labels_bruit])

plt.figure()
plt.scatter(X_imb[:, 0], X_imb[:, 1], c=y_imb, cmap="coolwarm", edgecolor="k", s=20)
plt.title("Jeu desequilibre avec points bruites pres de la frontiere")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.show()

X_imb_train, X_imb_test, y_imb_train, y_imb_test = train_test_split(
    X_imb, y_imb, test_size=0.3, random_state=1, stratify=y_imb)


### Exercice 7 (Master, ouvert)

Sur `X_imb_train`/`y_imb_train`/`X_imb_test`/`y_imb_test` préparés
ci-dessus, comparez, pour une même valeur de k de votre choix (par
exemple `k=15`), un `KNeighborsClassifier(weights='uniform')` et un
`KNeighborsClassifier(weights='distance')` : exactitude, et rappel de la
classe minoritaire (classe 1). La pondération par l'inverse de la
distance améliore-t-elle les résultats sur ce jeu bruité et
déséquilibré ? Justifiez avec les chiffres obtenus.

In [ ]:
k_choisi = 15

clf_uniforme = KNeighborsClassifier(n_neighbors=k_choisi, weights="uniform")
clf_uniforme.fit(X_imb_train, y_imb_train)
clf_pondere = KNeighborsClassifier(n_neighbors=k_choisi, weights="distance")
clf_pondere.fit(X_imb_train, y_imb_train)

pred_uniforme = clf_uniforme.predict(X_imb_test)
pred_pondere = clf_pondere.predict(X_imb_test)

acc_uniforme = accuracy_score(y_imb_test, pred_uniforme)
rappel_uniforme = recall_score(y_imb_test, pred_uniforme, pos_label=1)
acc_pondere = accuracy_score(y_imb_test, pred_pondere)
rappel_pondere = recall_score(y_imb_test, pred_pondere, pos_label=1)

print(f"Uniforme : exactitude={acc_uniforme:.3f}, rappel classe minoritaire={rappel_uniforme:.3f}")
print(f"Pondere  : exactitude={acc_pondere:.3f}, rappel classe minoritaire={rappel_pondere:.3f}")

if rappel_pondere >= rappel_uniforme:
    print("\n-> La ponderation par l'inverse de la distance tend a mieux detecter "
          "la classe minoritaire ici : les voisins tres proches pesent davantage "
          "que les points bruites plus lointains.")
else:
    print("\n-> Sur ce tirage, la ponderation n'ameliore pas le rappel : son effet "
          "depend fortement de la configuration locale du bruit et du choix de k.")


D'autres raffinements existent pour réduire le coût du k-NN « brut » sur
de grands jeux de données : structures d'indexation spatiale (arbres
kd, *ball trees*), inégalité triangulaire, ou sélection d'un
sous-ensemble représentatif de l'ensemble d'apprentissage (*Condensed
Nearest Neighbor*, Hart 1968). Le lien entre le 1-NN et le SVM (von
Luxburg et Bousquet, 2004) offre un autre éclairage sur les pertes de
substitution présentées en début de notebook — de quoi ouvrir le
prochain.